# Module 6 · The budget decision (profit per visit)

> **💬 Back to the Slack message:** *"Google converts better than Bing, mobile EPV is
> down, weekends are crashing — should we pull budget out of Bing and the weekend?"*

Now we answer it properly, end to end: pick the right metric, join the cost, quantify
with intervals, and convert it into a **recommendation with a decision rule and a
projected impact.** Runs on the **real NI `online_banking` data**.

The real objective isn't conversion rate *or* EPV — it's **profit per visit =
EPV − CPV**. That's what we optimize.

In [ ]:
import sys
from pathlib import Path
for _c in [Path.cwd(), *Path.cwd().parents]:
    if (_c / "src" / "ni_style.py").exists():
        sys.path.insert(0, str(_c / "src"))
        sys.path.insert(0, str(_c / ".claude" / "skills" / "_lib"))
        break
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import ni_style as ni      # chart style + palette
import ni_core as C        # the SAME primitives /budget-decision runs
ni.set_style()

visits = C.load_visits()          # one row per visit; revenue = EPV
cost = C.load_cost()              # AGGREGATE daily cost (no per-visit cost — the real constraint)
print(f"Loaded {len(visits):,} visits | {visits['date'].min().date()} -> {visits['date'].max().date()}")
visits.head()

## 1. The full funnel, by channel × platform — with uncertainty

One table an analyst can take to a meeting: volume, conversion (with CI), EPV (with
bootstrap CI), cost, **profit per visit (with CI)**, and ROAS. Cost is **aggregate**
(no per-visit cost), so CPV = summed cost ÷ visits — and the profit/visit interval is
the EPV interval shifted by that known CPV.

In [ ]:
def segment_stats(df):
    n = len(df); k = int(df.converted.sum())
    cr, cr_lo, cr_hi = C.wilson_ci(k, n)
    epv, epv_lo, epv_hi = C.bootstrap_mean_ci(df.revenue.values, n_boot=1500, seed=1)
    return pd.Series({"visits": n, "conv_rate": cr, "conv_lo": cr_lo, "conv_hi": cr_hi,
                      "EPV": epv, "EPV_lo": epv_lo, "EPV_hi": epv_hi, "rev": df.revenue.sum()})

seg = (visits.groupby(["channel","platform"], observed=True)
       .apply(segment_stats, include_groups=False).reset_index())

# The cost JOIN: aggregate daily cost summed by channel×platform → CPV = cost / visits
cost_by = cost.groupby(["channel","platform"])["cost"].sum().rename("cost_total")
seg = seg.merge(cost_by, on=["channel","platform"], how="left")
seg["CPV"] = seg.cost_total / seg.visits
seg["profit_visit"] = seg.EPV - seg.CPV
seg["ppv_lo"] = seg.EPV_lo - seg.CPV      # cost is known (aggregate) → shift the EPV interval
seg["ppv_hi"] = seg.EPV_hi - seg.CPV
seg["ROAS"] = seg.rev / seg.cost_total
seg = seg[seg.visits > 1500].sort_values("profit_visit", ascending=False)

show = seg.assign(
    conversion=lambda d: d.apply(lambda r: f"{r.conv_rate:.0%} [{r.conv_lo:.0%},{r.conv_hi:.0%}]", axis=1),
    EPV_=lambda d: d.EPV.map("${:.2f}".format),
    CPV_=lambda d: d.CPV.map("${:.2f}".format),
    profit=lambda d: d.apply(lambda r: f"${r.profit_visit:.2f} [${r.ppv_lo:.2f},${r.ppv_hi:.2f}]", axis=1),
    ROAS_=lambda d: d.ROAS.map("{:.1f}x".format),
)[["channel","platform","visits","conversion","EPV_","CPV_","profit","ROAS_"]]
show.columns = ["channel","platform","visits","conversion (95% CI)","EPV","CPV","profit/visit (95% CI)","ROAS"]
show.style.hide(axis="index").set_caption("NI funnel by channel × platform (real online_banking data)")

## 2. Where is the profit? A forest plot of profit-per-visit

Green = profitable to scale, red = losing money per visit. The CIs tell us which
calls are safe (interval clear of \$0) vs which need more data. Watch **Google mobile**:
it converts fine, but its interval sits **entirely below \$0** — it loses money.

In [ ]:
seg2 = seg.copy()
seg2["label"] = seg2.channel + " · " + seg2.platform
seg2 = seg2.sort_values("profit_visit")
fig, ax = plt.subplots(figsize=(11, 6.5))
for i, (_, r) in enumerate(seg2.iterrows()):
    col = ni.GREEN if r.profit_visit > 0 else ni.RED
    ax.plot([r.ppv_lo, r.ppv_hi], [i, i], color=col, lw=3)
    ax.plot(r.profit_visit, i, "o", color=col, ms=9)
ax.axvline(0, color=ni.NAVY, lw=1.6)
ax.set_yticks(range(len(seg2))); ax.set_yticklabels(seg2.label)
ax.set_xlabel("profit per visit ($, 95% CI)")
ni.titlebox(ax, "Profit per visit by segment — what to scale, what to cut",
            "intervals clear of $0 are safe bets; Google mobile is a confident money-loser")
fig.tight_layout(); ni.savefig(fig, "m6_profit_forest"); plt.show()

## 3. A decision rule, and the projected impact

Unlike a toy dataset, here there **is** a loser: Google mobile's profit/visit CI is
entirely below \$0. So the rule has two parts — **cut the confirmed loser**, and
**move its budget to the best paid segment**. We bid only on **paid** channels
(Organic is near-free SEO — protect it, but it's not a bid lever), so we rank the paid
segments by **profit per visit** and reallocate.

**Rule:** *Scale up* the top third of paid segments; *trim* the bottom third
(their profit/visit CI sits clearly below the leaders — or below \$0). Then simulate
shifting 20% of the weakest segments' spend into the strongest.

In [ ]:
# We control bids on PAID channels only (Organic is near-free SEO, not a budget lever)
paid = seg[seg.channel != "Organic"].copy().sort_values("profit_visit")
lo_thr, hi_thr = paid.profit_visit.quantile([1/3, 2/3])
paid["verdict"] = np.where(paid.profit_visit >= hi_thr, "SCALE UP",
                   np.where(paid.profit_visit <= lo_thr, "TRIM", "hold / optimize"))
print("Budget decision for PAID segments (we don't bid on Organic):\n")
print(paid[["channel","platform","visits","profit_visit","ppv_lo","ppv_hi","verdict"]]
      .round(2).sort_values("profit_visit", ascending=False).to_string(index=False))

# Reallocate 20% of the weakest ('TRIM') segments' traffic into the strongest segment
winners = paid[paid.verdict == "SCALE UP"]
losers  = paid[paid.verdict == "TRIM"]
top = winners.iloc[winners.profit_visit.values.argmax()]
moved_visits = int(0.20 * losers.visits.sum())
l_ppv = (losers.profit_visit * losers.visits).sum() / losers.visits.sum()
delta = moved_visits * (top.profit_visit - l_ppv)
print(f"\nMove ~{moved_visits:,} visits/qtr from TRIM segments (avg ${l_ppv:.2f}/visit) "
      f"to {top.channel}·{top.platform} (${top.profit_visit:.2f}/visit)")

current = (seg.profit_visit * seg.visits).sum()
fig, ax = plt.subplots(figsize=(8.5, 5))
ax.bar(["current\n(quarter)","after 20%\nreallocation"], [current, current+delta],
       color=[ni.GREY, ni.GREEN])
for i, v in enumerate([current, current+delta]):
    ax.text(i, v, f"${v:,.0f}", ha="center", va="bottom", fontweight="bold", color=ni.NAVY)
ax.set_ylim(0, (current+delta)*1.12); ax.set_ylabel("modeled total profit ($ / quarter)")
ni.titlebox(ax, "Projected impact of the reallocation",
            f"shift 20% of the weakest paid traffic to the top segment → +${delta:,.0f} / quarter")
fig.tight_layout(); ni.savefig(fig, "m6_reallocation"); plt.show()

## 4. The answer to the Slack message

> **"Google converts better than Bing — should we pull budget out of Bing?"**
>
> 🚫 **No — that's backwards.** Bing is your **most profitable paid traffic**
> (profit/visit +$0.54 desktop, +$0.39 mobile, both CIs clear of \$0). Conversion
> rate is the wrong yardstick: Google *desktop* has the **best** conversion (11.9%)
> yet barely breaks even, because its clicks are expensive.
>
> ✅ **The real money-loser is Google *mobile*.** It converts fine (9%), but CPV > EPV,
> so profit/visit is **−$0.21 with a CI entirely below \$0**. That is the segment to
> trim — and moving its budget to Bing desktop is the biggest single lever.
>
> ✅ **On mobile EPV:** real — within every channel, mobile EPV sits below desktop.
> Trim mobile bids where profit/visit is weakest and re-measure.
>
> 🚫 **Ignore the "weekend crash":** it's a **traffic-mix / seasonality artifact**
> (Module 5) — put any day-over-day move against the monthly band before reacting.
>
> 💡 **The biggest lever isn't paid reallocation at all:** *Organic* earns ~**54× ROAS**
> at near-zero cost — by far the most profitable traffic. **Protect and grow SEO.**

## 🧾 The Statistical Decision Checklist (pin this at your desk)

| Step | Ask yourself |
|---|---|
| **1. Trust the average** | Is the mean even typical? Skew, zeros, whales? (M1 · /profile-data) |
| **2. Effective-n** | Are rows independent, or clustered? How much data is *really* here? (M2) |
| **3. Is it real** | Right test for the shape? The n-trap? A confound / Simpson's? (M3 · /significance-check) |
| **4. Borrow strength** | On a thin slice, what does a prior from history say? (M4 · /bayesian-update) |
| **5. Over time** | Is the move outside the monthly band / a regime break? (M5 · /trend-check) |
| **6. Decide** | Does it change **profit per visit**? What's the rule and projected impact? (M6 · /budget-decision) |

> **The one-liner:** *Significance tells you something is there; the confidence
> interval tells you whether it's worth acting on; profit-per-visit tells you what to do.*

### 🎓 That's the concept arc — from a Slack message to a defensible decision.

**→ The skill that automates this:** `/budget-decision`. Next: **Module 7 — compose the multi-agent flow** that runs all of this as one gated procedure.